# 2. Clustering (Aya)

- **Goal:** Automatic Market Segmentation.
- **Implementation:** Use K-Means or DBSCAN on numeric features (`RAM_GB`, `TOTAL_PIXELS`, `PPI`).
- **Insight:** Discover if the market naturally groups into segments like "Gaming," "Ultrabooks," or "Old Office Tech" and how price varies within those clusters.

### 📊 Evaluation Metrics
- **Silhouette Score**: Measures how similar an object is to its own cluster (cohesion) compared to other clusters (separation).

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# Set Style
sns.set_style("whitegrid")

### 1. Load Data
Using the training dataset (70% split) for analysis.

In [ ]:
try:
    df = pd.read_csv('training_dataset.csv')
except FileNotFoundError:
    df = pd.read_csv('06_Model_Training_Evaluation/training_dataset.csv')

print(f"Data Shape: {df.shape}")
df.head()

### 2. Feature Selection & Preprocessing
Target features for clustering: `RAM_GB`, `TOTAL_PIXELS`, `PPI`.

In [ ]:
features = ['RAM_GB', 'TOTAL_PIXELS', 'PPI']
X = df[features].copy()

# Drop missing if any
X.dropna(inplace=True)
df_clean = df.loc[X.index].copy()

# Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Features Scaled.")

### 3. Clustering (K-Means)
We will test multiple K values to find the optimal number of clusters.

In [ ]:
results = []
k_values = range(2, 7)

for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    results.append({'K': k, 'Silhouette': score})
    print(f"K={k}, Silhouette Score={score:.4f}")

best_result = max(results, key=lambda x: x['Silhouette'])
print(f"\nBest K: {best_result['K']} with Score: {best_result['Silhouette']:.4f}")

### 4. Cluster Analysis & Interpretation

In [ ]:
best_k = best_result['K']
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df_clean['Cluster'] = kmeans.fit_predict(X_scaled)

# Visualize relationships
summary = df_clean.groupby('Cluster')[features + ['PRICE']].mean()
summary['Count'] = df_clean['Cluster'].value_counts()
summary

### 5. Visualization (PCA)

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(10, 6))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=df_clean['Cluster'], palette='viridis', s=100)
plt.title(f'Market Clusters (K={best_k}) - PCA Projection')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend(title='Cluster')
plt.show()